# 통합수업일지 × 로또 확률 게임 × 「데이터 분석 기법 더하기」 10종

**실습 1개 노트북:** 과거 949~1248회(300회차) 번호·판매액·당첨금을 사용합니다. 실제 사용자 제공 CSV의 부분 추출이며 최신 정보가 아닙니다. 지역·고객·댓글 데이터는 별도 가상 예제입니다.

**검증 원칙:** 역순 누출 금지 · 구체적 질문 → 데이터 → 계산 → 해석 → 한계 · 다음 로또번호에 대한 예측력 주장이 아닙니다.

In [ ]:
from pathlib import Path
from math import comb
import numpy as np, pandas as pd, json
from IPython.display import display,Image
P=Path.cwd();d=pd.read_csv(P/'data/lotto_949_1248_sample.csv').sort_values('drwNo')
prize=pd.read_csv(P/'data/lotto_949_1248_sales_prize.csv').sort_values('drwNo')
r=json.loads((P/'data/analysis_results.json').read_text(encoding='utf-8'))
assert len(d)==len(prize)==300 and d.drwNo.tolist()==list(range(949,1249))
print('300회차 과거 추첨자료 검증 완료:',d.drwNo.iloc[0],d.drwNo.iloc[-1])

## ① 빈도와 최근 60회
**질문:** 과거에 자주 나온 번호는? **결과:** 300회차 6개 번호 총 1,800회 출현. 공정한 추첨에서 번호별 기대 40회. 빈도가 높다고 다음 회차 확률이 높아지는 것은 아닙니다.

In [ ]:
a=d[[f'n{i}' for i in range(1,7)]].to_numpy()
f=np.bincount(a.ravel(),minlength=46)[1:]
assert f.sum()==1800
print('전체 상위번호',sorted(range(1,46),key=lambda n:(-f[n-1],n))[:6])
print('최근60회 상위번호',sorted(range(1,46),key=lambda n:(-r['frequency']['last60'][n-1],n))[:6])

## ② 연속·짝·홀·합계·동반출현
**질문:** 연속번호가 있는 추첨은 드문가? **결과:** 154/300회(51.33%)에 연속번호가 있었고, 수학적 이론값은 52.87%입니다.

In [ ]:
from itertools import combinations
from collections import Counter
cons=sum(any(b-a==1 for a,b in zip(sorted(x),sorted(x)[1:])) for x in a)
pairs=Counter(tuple(z) for row in a for z in combinations(sorted(row),2))
print('연속번호 포함:',cons,'/300; 이론%',round(100*(1-comb(40,6)/comb(45,6)),2))
print('합계 평균:',round(a.sum(axis=1).mean(),3),'주요 번호쌍:',pairs.most_common(3))

## ③ 확률·Monte Carlo
**질문:** 한 특정 조합의 1등 확률은? **결과:** 1/8,145,060. 무작위 선택 1게임의 예상 일치 번호 수는 0.8개입니다.

In [ ]:
print('45C6=',comb(45,6))
print('1등 확률=',1/comb(45,6),'기대일치수=',6*6/45)
print('600회 시뮬레이션에서 계산된 번호빈도 통계량 및 MC p값=',r['probability']['mc_stat'],r['probability']['mc_pvalue'])

## ④ 시간순 백테스트
**질문:** 과거만으로 선택한 빈도 규칙이 무작위보다 나았나? **결과:** 1099~1248회(150회) 이번 표본은 최근빈도 0.773, 지수가중 0.707, 무작위 0.913개입니다. 확률 개선이 입증되지 않았습니다.

In [ ]:
b=r['backtest']['strategies']
for method,x in b.items():print(method,'회차',x['draws'],'평균일치',x['mean_matches'],'3개이상',x['hits_ge_3'])
assert all(x['hits_6']==0 for x in b.values())

## ⑤ PDF 추가기법 1~4: 로그·IQR·StandardScaler·검정
**질문:** 1등 지급액의 분포·이상치·기간별 평균차는? **결과:** IQR 이상치 후보 10건, Welch 검정 p≈0.003205. 실제 원인은 이 분석만으로 단정할 수 없습니다.

In [ ]:
from scipy.stats import ttest_ind
from sklearn.preprocessing import StandardScaler,MinMaxScaler
q1,q3=prize.firstWinamnt.quantile([.25,.75]);lo,hi=q1-1.5*(q3-q1),q3+1.5*(q3-q1)
print('IQR 후보:',sum((prize.firstWinamnt<lo)|(prize.firstWinamnt>hi)))
z=StandardScaler().fit_transform(prize[['firstWinamnt','totSellamnt']]);print('표준화 후 평균=',z.mean(axis=0))
print('Welch 검정',ttest_ind(prize.firstWinamnt.iloc[:150],prize.firstWinamnt.iloc[150:],equal_var=False))

## ⑥ PDF 추가기법 5~8: 시계열·10만명당 비율·RFM/코호트·지도
**질문:** 시간·분모·코호트의 차이를 어떻게 다루나? **결과:** 2026년 10월 회차당 평균 판매액은 전년 동월 대비 약 -0.76%. 가상 지역 A/B의 원건수와 10만명당 순위는 다릅니다.

In [ ]:
monthly=prize.assign(date=pd.to_datetime(prize.drwNoDate)).groupby(pd.to_datetime(prize.drwNoDate).dt.to_period('M')).totSellamnt.mean()
print('최근 월 전년 동월 비율:',round((monthly.iloc[-1]/monthly.iloc[-13]-1)*100,4),'%')
c=pd.read_csv(P/'data/synthetic_regions.csv');display(c[['region','event_count','per_100k']])
o=pd.read_csv(P/'data/synthetic_orders.csv');print('가상 고객 수',o.customer_id.nunique(),'구매 기록',len(o))

## ⑦ PDF 추가기법 9~10: Plotly와 TF-IDF
**질문:** 실측 산점도를 마우스로 보고, 가상 불편 댓글의 특징어는? **결과:** 공개 Plotly 차트와 가상 댓글 '오류'의 높은 평균 TF-IDF를 확인했습니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
reviews=pd.read_csv(P/'data/synthetic_comments.csv')
v=TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
m=v.fit_transform(reviews.text);mask=(reviews.label=='불편').to_numpy()
diff=np.asarray(m[mask].mean(axis=0)).ravel()-np.asarray(m[~mask].mean(axis=0)).ravel()
print('가상 불편 댓글 특징어',[(v.get_feature_names_out()[i],round(float(diff[i]),3)) for i in np.argsort(-diff)[:4]])
print('Plotly 그래프는 plotly_interactive.html 파일에서 열기')

## ⑧ 과제 제출물과 최종 검수
**실제 결과:** GitHub의 `analysis.py`는 CSV를 읽어 모든 결과 JSON·8개 PNG를 재생성합니다. ZIP에는 8개 PNG와 실행 출력이 저장된 노트북도 있습니다.

**주의:** K-pop/Olist와 로또 결과를 섞지 않습니다. 당첨금·판매액·추첨번호는 과거 기록이며 가상 지역·가상 고객·가상 댓글은 로또 실측 자료가 아닙니다. 균등 추첨에서는 어떤 6개 조합도 다른 조합보다 1등 확률이 높지 않습니다.